# 03 · Videos de los agentes
**Autor:** Daniel Sebastian Velasco Munar · Aprendizaje Profundo 202630 · PUJ

Se graban episodios cortos en el simulador para ver cómo se mueve cada agente:

- Decision Transformer (semilla 1, ya entrenado en el notebook 01) con los conjuntos expert de Hopper y HalfCheetah.
- El mismo Decision Transformer de Hopper pidiéndole solo el 25 % del retorno experto.
- Random Forest entrenado con los mismos datos expert, que es el caso en el que la imitación falla.

Los GIF quedan en `Mi unidad/proyecto_decision_transformer/gifs`.

> **Antes de ejecutar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`. Tarda unos 15 minutos.

In [ ]:
!pip install -q "gymnasium[mujoco]" mujoco
!pip install -q "minari[all]"
!pip install -q transformers imageio

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
os.environ["MUJOCO_GL"] = "egl"      # render sin pantalla en Colab
import numpy as np, torch, minari, gymnasium as gym, imageio
from sklearn.ensemble import RandomForestRegressor
from transformers import DecisionTransformerConfig, DecisionTransformerModel
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from IPython.display import Image, display
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/proyecto_decision_transformer"
GIFS = f"{OUT}/gifs"; os.makedirs(GIFS, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REFS = json.load(open(f"{OUT}/referencias.json"))
K, RTG_SCALE, MAX_EP_LEN = 20, 1000.0, 1000
PASOS_VIDEO = 700        # pasos grabados por episodio (de 1 000 posibles)
SEMILLA_DT = 1
normalized = lambda env_name, R: 100 * (R - REFS[env_name]["random"]) / (REFS[env_name]["expert"] - REFS[env_name]["random"])
print(DEVICE)

In [ ]:
class DTAgent:
    # Igual que en los notebooks 01 y 02
    def __init__(self, model, mean, std, sd, ad, target):
        self.model, self.mean, self.std, self.sd, self.ad, self.target = model, mean, std, sd, ad, target
    def reset(self):
        self.S, self.A, self.R = [], [], []; self.G = [self.target / RTG_SCALE]; self.t = 0
    @torch.no_grad()
    def act(self, obs):
        sd, ad = self.sd, self.ad
        self.S.append(((obs - self.mean) / self.std).astype(np.float32)); self.A.append(np.zeros(ad, np.float32)); self.R.append(0.0)
        L = min(K, len(self.S)); pad = K - L
        def seq(x, dim):
            x = np.asarray(x[-L:], np.float32).reshape(L, dim)
            return np.concatenate([np.zeros((pad, dim), np.float32), x])
        ts = np.concatenate([np.zeros(pad, np.int64), np.minimum(np.arange(self.t - L + 1, self.t + 1), MAX_EP_LEN - 1)])
        m = np.concatenate([np.zeros(pad), np.ones(L)]).astype(np.float32)
        to = lambda x: torch.tensor(x[None], device=DEVICE)
        out = self.model(states=to(seq(self.S, sd)), actions=to(seq(self.A, ad)), rewards=to(seq(self.R, 1)),
                         returns_to_go=to(seq(self.G, 1)), timesteps=to(ts), attention_mask=to(m), return_dict=True)
        a = out.action_preds[0, -1].cpu().numpy(); self.A[-1] = a
        return np.clip(a, -1, 1)
    def update(self, r):
        self.R[-1] = r; self.G.append(self.G[-1] - r / RTG_SCALE); self.t += 1

class SimpleAgent:
    def __init__(self, fn): self.fn = fn
    def reset(self): pass
    def update(self, r): pass
    def act(self, obs): return np.clip(self.fn(obs), -1, 1)

def load_dt(env_name, q, target_frac=1.0):
    ck = torch.load(f"{OUT}/modelos/dt_{env_name}_{q}_{SEMILLA_DT}.pt", map_location=DEVICE, weights_only=False)
    cfg = DecisionTransformerConfig.from_dict(ck["config"])
    model = DecisionTransformerModel(cfg).to(DEVICE); model.load_state_dict(ck["pesos"]); model.eval()
    return DTAgent(model, ck["media"], ck["desv"], cfg.state_dim, cfg.act_dim, target_frac * REFS[env_name]["expert"])

def train_rf(ds):
    # Igual que en el notebook 01: 100 000 transiciones al azar, 100 árboles
    X = np.concatenate([ep.observations[:-1] for ep in ds.iterate_episodes()]).astype(np.float32)
    Y = np.concatenate([ep.actions for ep in ds.iterate_episodes()]).astype(np.float32)
    idx = np.random.default_rng(1).choice(len(X), size=100_000, replace=False)
    rf = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=1)
    rf.fit(X[idx], Y[idx]); rf.set_params(n_jobs=1)
    return SimpleAgent(lambda o: rf.predict(o[None])[0])

def grabar(ds, agent, nombre, env_name, seed=10_000):
    # cámara "track" del modelo: sigue al robot para que no se salga del cuadro
    env = gym.make(ds.env_spec, render_mode="rgb_array", width=360, height=270, camera_name="track")
    obs, _ = env.reset(seed=seed); agent.reset(); frames, R, t = [], 0.0, 0
    for t in range(PASOS_VIDEO):
        obs, r, term, trunc, _ = env.step(agent.act(obs)); agent.update(r); R += r
        if t % 4 == 0:
            frames.append(env.render())
        if term or trunc:
            break
    env.close()
    path = f"{GIFS}/{nombre}.gif"
    imageio.mimsave(path, frames, fps=25, loop=0)
    estado = "se cayó" if term else "sigue en pie"
    print(f"{nombre:<28} {t + 1:>4} pasos · retorno {R:8.1f} · {estado} · {os.path.getsize(path) / 1e6:.1f} MB")
    display(Image(filename=path))

In [ ]:
t0 = time.time()
for env_name in ["hopper", "halfcheetah"]:
    ds = minari.load_dataset(f"mujoco/{env_name}/expert-v0", download=True)
    grabar(ds, load_dt(env_name, "expert"), f"{env_name}_dt", env_name)
    if env_name == "hopper":
        grabar(ds, load_dt(env_name, "expert", 0.25), f"{env_name}_dt_25", env_name)
    grabar(ds, train_rf(ds), f"{env_name}_random_forest", env_name)
    print(f"{(time.time() - t0) / 60:.1f} min")
print("Listo:", sorted(os.listdir(GIFS)))

In [ ]:
# Libera la GPU al terminar para no gastar unidades
from google.colab import runtime
runtime.unassign()